# D_Gold - Modelo estrella

Gold consume las tablas Silver registradas en Unity Catalog y construye
4 dimensiones y 1 tabla de hechos, listas para Power BI.


In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

customers = spark.table(f"{SILVER_SCHEMA}.customers")


def write_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


# Dimensión clientes
df_dim_customers = (
    customers
    .withColumn(
        "gender_customer",
        F.when(F.col("gender") == "Male", "M")
         .when(F.col("gender") == "Female", "F")
         .otherwise(F.col("gender"))
    )
    .withColumn("year_join", F.year("join_date"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .drop("gender", "loyalty_member")
)
write_gold(df_dim_customers, f"{GOLD_SCHEMA}.dim_customers")


print("GOLD - tabla dimensional dim_customers generado correctamente")
print("Dimensiones: dim_customers")


In [0]:
%sql

SELECT 'dim_customers' AS tabla, COUNT(*) AS total_filas FROM workspace.gold.dim_customers

